# VAE sur CIFAR10 – 4 classes (numéro étudiant 9439)

**Tâche :**
- Charger CIFAR10, garder 4 classes correspondant aux 4 chiffres différents du numéro étudiant 9439 → classes **0, 3, 4, 9** (avion, chat, cerf, camion).
- Choisir une dimension latente, créer et entraîner un VAE.
- Choisir 2 images de 2 classes différentes, obtenir leurs points dans l’espace latent, faire une transformation (interpolation) entre elles et visualiser les images de transition.

## 1. Imports et configuration

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers, models, datasets, callbacks, losses, optimizers, metrics

print("Keras:", keras.__version__, "| Backend:", keras.backend.backend())

## 2. Chargement CIFAR10 et filtrage des 4 classes

Numéro étudiant **9439** → quatre chiffres différents : **9, 4, 3**. Pour avoir 4 classes on prend **0, 3, 4, 9** (CIFAR10 : 0=avion, 3=chat, 4=cerf, 9=camion).

In [ ]:
# Classes correspondant aux chiffres 0, 3, 4, 9 (9439 → 9,4,3 + 0 pour avoir 4 classes)
SELECTED_CLASSES = [0, 3, 4, 9]  # airplane, cat, deer, truck
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
SELECTED_NAMES = [CLASS_NAMES[i] for i in SELECTED_CLASSES]
print("Classes utilisées:", SELECTED_CLASSES, "→", SELECTED_NAMES)

(x_train_full, y_train_full), (x_test_full, y_test_full) = datasets.cifar10.load_data()

# Flatten labels if needed
if len(y_train_full.shape) > 1:
    y_train_full = y_train_full.flatten()
if len(y_test_full.shape) > 1:
    y_test_full = y_test_full.flatten()

# Garder uniquement les images des 4 classes
mask_train = np.isin(y_train_full, SELECTED_CLASSES)
mask_test = np.isin(y_test_full, SELECTED_CLASSES)
x_train = x_train_full[mask_train].astype("float32") / 255.0
y_train = y_train_full[mask_train]
x_test = x_test_full[mask_test].astype("float32") / 255.0
y_test = y_test_full[mask_test]

print("Forme x_train:", x_train.shape, "| y_train:", y_train.shape)
print("Forme x_test:", x_test.shape, "| y_test:", y_test.shape)

## 3. Paramètres du VAE et couche Sampling

In [ ]:
IMAGE_SIZE = 32
CHANNELS = 3
LATENT_DIM = 64   # dimension de l'espace latent
BATCH_SIZE = 64
EPOCHS = 30
BETA = 1.0


class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = keras.random.normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

## 4. Encodeur (32x32x3 → latent)

In [ ]:
encoder_input = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS), name="encoder_input")
x = layers.Conv2D(32, (3, 3), strides=2, activation="relu", padding="same")(encoder_input)
x = layers.Conv2D(64, (3, 3), strides=2, activation="relu", padding="same")(x)
x = layers.Conv2D(128, (3, 3), strides=2, activation="relu", padding="same")(x)
shape_before_flattening = x.shape[1:]
x = layers.Flatten()(x)
z_mean = layers.Dense(LATENT_DIM, name="z_mean")(x)
z_log_var = layers.Dense(LATENT_DIM, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])
encoder = models.Model(encoder_input, [z_mean, z_log_var, z], name="encoder")
encoder.summary()

## 5. Décodeur (latent → 32x32x3)

In [ ]:
decoder_input = layers.Input(shape=(LATENT_DIM,), name="decoder_input")
x = layers.Dense(np.prod(shape_before_flattening))(decoder_input)
x = layers.Reshape(shape_before_flattening)(x)
x = layers.Conv2DTranspose(128, (3, 3), strides=2, activation="relu", padding="same")(x)
x = layers.Conv2DTranspose(64, (3, 3), strides=2, activation="relu", padding="same")(x)
x = layers.Conv2DTranspose(32, (3, 3), strides=2, activation="relu", padding="same")(x)
decoder_output = layers.Conv2D(CHANNELS, (3, 3), activation="sigmoid", padding="same", name="decoder_output")(x)
decoder = models.Model(decoder_input, decoder_output)
decoder.summary()

## 6. Modèle VAE et entraînement

In [ ]:
class VAE(models.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.reconstruction_loss_tracker, self.kl_loss_tracker]

    def call(self, inputs):
        z_mean, z_log_var, z = self.encoder(inputs)
        reconstruction = self.decoder(z)
        return z_mean, z_log_var, reconstruction

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, reconstruction = self(data)
            reconstruction_loss = tf.reduce_mean(
                BETA * losses.binary_crossentropy(data, reconstruction, axis=(1, 2, 3))
            )
            kl_loss = tf.reduce_mean(
                tf.reduce_sum(
                    -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)),
                    axis=1,
                )
            )
            total_loss = reconstruction_loss + kl_loss
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        if isinstance(data, tuple):
            data = data[0]
        z_mean, z_log_var, reconstruction = self(data)
        reconstruction_loss = tf.reduce_mean(
            BETA * losses.binary_crossentropy(data, reconstruction, axis=(1, 2, 3))
        )
        kl_loss = tf.reduce_mean(
            tf.reduce_sum(
                -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)),
                axis=1,
            )
        )
        return {"loss": reconstruction_loss + kl_loss, "reconstruction_loss": reconstruction_loss, "kl_loss": kl_loss}


vae = VAE(encoder, decoder)
vae.compile(optimizer=optimizers.Adam(learning_rate=1e-3))

In [ ]:
history = vae.fit(
    x_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_data=(x_test, x_test),
)
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['total_loss'], label='train total')
plt.plot(history.history['val_loss'], label='val total')
plt.legend()
plt.title('Loss')
plt.subplot(1, 2, 2)
plt.plot(history.history['reconstruction_loss'], label='train recon')
plt.plot(history.history['kl_loss'], label='train KL')
plt.legend()
plt.title('Reconstruction & KL')
plt.tight_layout()
plt.show()

## 7. Deux images de deux classes différentes et points dans l’espace latent

In [ ]:
# Choisir 2 classes différentes (ex: 0=avion et 9=camion)
class_a, class_b = SELECTED_CLASSES[0], SELECTED_CLASSES[-1]  # 0 et 9
idx_a = np.where(y_test == class_a)[0][0]
idx_b = np.where(y_test == class_b)[0][0]
img_a = x_test[idx_a : idx_a + 1]  # (1, 32, 32, 3)
img_b = x_test[idx_b : idx_b + 1]

_, _, z_a = encoder.predict(img_a, verbose=0)
_, _, z_b = encoder.predict(img_b, verbose=0)
z_a = z_a[0]
z_b = z_b[0]

print("Classe A:", class_a, CLASS_NAMES[class_a], "| Classe B:", class_b, CLASS_NAMES[class_b])
print("Points latents z_a shape:", z_a.shape, "| z_b shape:", z_b.shape)

## 8. Transformation (interpolation) et visualisation des images de transition

In [ ]:
n_steps = 10
factors = np.linspace(0, 1, n_steps)

fig, axes = plt.subplots(1, n_steps + 2, figsize=(n_steps + 2, 2))

axes[0].imshow(img_a[0])
axes[0].set_title(CLASS_NAMES[class_a])
axes[0].axis("off")

for i, t in enumerate(factors):
    z_interp = (1 - t) * z_a + t * z_b
    decoded = decoder.predict(z_interp[np.newaxis, :], verbose=0)[0]
    axes[i + 1].imshow(np.clip(decoded, 0, 1))
    axes[i + 1].set_title(f"t={t:.2f}")
    axes[i + 1].axis("off")

axes[-1].imshow(img_b[0])
axes[-1].set_title(CLASS_NAMES[class_b])
axes[-1].axis("off")
plt.suptitle(f"Transition latent: {CLASS_NAMES[class_a]} → {CLASS_NAMES[class_b]}")
plt.tight_layout()
plt.show()